# PMSD baseline — synthetic logs

All 10 synthetic datasets (`trim='none'`), fitting on train+val, simulating `concurrent_cases`/`throughput_time` over the test horizon. See `README.md` for design decisions.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import pm4py


ROOT = Path.cwd().resolve().parent.parent
PMSD_MAIN = ROOT / "simulation_baselines" / "PMSD-main"
sys.path.insert(0, str(PMSD_MAIN))
sys.path.insert(0, str(ROOT))

from pmsd import run_pmsd_synthetic

DATA_DIR = ROOT / "data" / "synthetic"
SYNTH_DATASETS = sorted(
    p.stem for p in DATA_DIR.glob("*.xes") if "recency" not in p.stem
)
print(f"{len(SYNTH_DATASETS)} datasets:", SYNTH_DATASETS)

In [ ]:
results = {}
for name in SYNTH_DATASETS:
    print(f"=== {name} ===")
    log = pm4py.read_xes(str(DATA_DIR / f"{name}.xes"))
    results[name] = run_pmsd_synthetic(name, log, save=True)
    print("  ", results[name]["metrics"])

## Summary table

In [ ]:
summary = pd.DataFrame([
    {"dataset": name, **res["metrics"]}
    for name, res in results.items()
]).set_index("dataset")
summary

## Per-dataset actual-vs-predicted plots

In [ ]:
for name, res in results.items():
    sim = res["simulated"]
    cc_actual = res["cc_test"].reindex(sim.index)
    tt_actual = res["tt_test"].reindex(sim.index)

    fig, axes = plt.subplots(1, 2, figsize=(14, 3.5))
    axes[0].plot(cc_actual.index, cc_actual.values, label="actual", color="green")
    axes[0].plot(sim.index, sim["concurrent_cases"], label="PMSD", color="steelblue", ls="--")
    axes[0].set_title(f"{name} — concurrent_cases (MAE={res['metrics']['cc_mae']:.1f})")
    axes[0].legend(fontsize=8)

    axes[1].plot(tt_actual.index, tt_actual.values, label="actual", color="green")
    axes[1].plot(sim.index, sim["throughput_time"], label="PMSD", color="darkorange", ls="--")
    axes[1].set_title(f"{name} — throughput_time (MAE={res['metrics']['tt_mae']:.1f})")
    axes[1].legend(fontsize=8)

    plt.tight_layout()
    plt.show()

## Discovered relations (informational, all 9 SD-Log columns)

In [ ]:
for name, res in results.items():
    print(f"=== {name} ===")
    for target, candidates in res["relations"].items():
        top = ", ".join(f"{p}(shift={s}, corr={c:.2f})" for p, s, c in candidates[:3])
        print(f"  {target} <- {top if top else '(none above threshold)'}")

# PMSD baseline — real-life logs

All 8 real-life datasets x all 7 trims (56 combos). `service_time`/`waiting_time`/`process_active_time` are `0.0` for real-life (no start timestamps in these logs) -- unused by the simulated series (`concurrent_cases`/`arrival_rate`/`finish_rate`/`throughput_time`).

In [ ]:
from time import time as _time

REAL_DATA_DIR = ROOT / "data" / "real-life"
REAL_DATASETS = ["bpic12-a", "bpic15-1", "bpic15-2", "bpic17-o", "bpic20-dom", "bpic20-int", "helpdesk", "sepsis"]
REAL_TRIM_NAMES = ["none", "peak_0.6", "peak_0.7", "peak_0.8", "magnitude_1", "magnitude_2", "magnitude_3"]

from pmsd import run_pmsd

print(f"{len(REAL_DATASETS)} datasets x {len(REAL_TRIM_NAMES)} trims = {len(REAL_DATASETS) * len(REAL_TRIM_NAMES)} combos")

In [ ]:
real_logs = {}
for name in REAL_DATASETS:
    real_logs[name] = pm4py.read_xes(str(REAL_DATA_DIR / f"{name}.xes"))
    print(name, len(real_logs[name]), "events")

In [ ]:
real_results = {}
_t0 = _time()
for name in REAL_DATASETS:
    for trim in REAL_TRIM_NAMES:
        key = (name, trim)
        try:
            real_results[key] = run_pmsd(name, real_logs[name], trim=trim, is_real=True, save=True)
            print(f"{name:12s} {trim:12s}", real_results[key]["metrics"])
        except Exception as e:
            print(f"{name:12s} {trim:12s} FAILED: {e}")
print(f"done in {_time()-_t0:.0f}s, {len(real_results)}/{len(REAL_DATASETS)*len(REAL_TRIM_NAMES)} combos succeeded")

## Real-life summary table

In [ ]:
real_summary = pd.DataFrame([
    {"dataset": name, "trim": trim, **res["metrics"]}
    for (name, trim), res in real_results.items()
]).set_index(["dataset", "trim"]).sort_index()
real_summary

## Real-life: mean MAE by trim (across datasets)

In [ ]:
real_summary.reset_index().groupby("trim")[["cc_mae", "tt_mae"]].mean().reindex(REAL_TRIM_NAMES)

## Per-dataset actual-vs-predicted plots (`trim='peak_0.7'`)

In [ ]:
for name in REAL_DATASETS:
    key = (name, "peak_0.7")
    if key not in real_results:
        continue
    res = real_results[key]
    sim = res["simulated"]
    cc_actual = res["cc_test"].reindex(sim.index)
    tt_actual = res["tt_test"].reindex(sim.index)

    fig, axes = plt.subplots(1, 2, figsize=(14, 3.5))
    axes[0].plot(cc_actual.index, cc_actual.values, label="actual", color="green")
    axes[0].plot(sim.index, sim["concurrent_cases"], label="PMSD", color="steelblue", ls="--")
    axes[0].set_title(f"{name} / peak_0.7 — concurrent_cases (MAE={res['metrics']['cc_mae']:.1f})")
    axes[0].legend(fontsize=8)

    axes[1].plot(tt_actual.index, tt_actual.values, label="actual", color="green")
    axes[1].plot(sim.index, sim["throughput_time"], label="PMSD", color="darkorange", ls="--")
    axes[1].set_title(f"{name} / peak_0.7 — throughput_time (MAE={res['metrics']['tt_mae']:.1f})")
    axes[1].legend(fontsize=8)

    plt.tight_layout()
    plt.show()